# Race Conditions in OpenMP

A **race condition** occurs when two or more threads access shared memory simultaneously, and at least one of them is writing. The result depends on the unpredictable order of execution - making bugs hard to reproduce and debug.

## 1. The Classic Race Condition - Parallel Counter

We want to increment a shared counter 1 000 000 times using multiple threads.
Intuitively `counter++` looks one action, but it compiles to **three** instructions:

```
LOAD   reg ← counter     ; read current value
ADD    reg, 1            ; increment in register
STORE  counter ← reg     ; write back
```

If two threads execute these steps concurrently, one write will silently overwrite the other - an **increment is lost**.

In [1]:
#include <omp.h>
#include <stdio.h>
#define N 1000000

void example1()
{
    int counter = 0;
    
    // Run the experiment 5 times to show non-determinism
    for (int run = 0; run < 5; run++) {
        counter = 0;
        #pragma omp parallel for num_threads(4)
        {
            for (int i = 0; i < N; i++) {
                counter++;   // RACE CONDITION: no protection
            }
        }
        printf("Run %d: counter = %d  (expected %d, lost %d)\n",
               run+1, counter, N, N - counter);
    }
}

example1();

Run 1: counter = 458255  (expected 1000000, lost 541745)
Run 2: counter = 340448  (expected 1000000, lost 659552)
Run 3: counter = 367539  (expected 1000000, lost 632461)
Run 4: counter = 367803  (expected 1000000, lost 632197)
Run 5: counter = 405751  (expected 1000000, lost 594249)


### What you should observe

- The counter is **almost never** 1 000 000.
- Each run gives a **different** wrong answer - the race is truly non-deterministic.

> **Key insight**: You cannot rely on "it worked in my tests" for racy code. On a lightly loaded machine you might get lucky; under production load the bug emerges.

---
## 2. Fix #1 - `#pragma omp critical`

`critical` creates a **named mutex**: only one thread at a time may execute the enclosed block. It is the most general but also the most expensive fix.

In [2]:
#include<time.h>

void example2() {
    int counter = 0;
    double t0 = omp_get_wtime();

    #pragma omp parallel for num_threads(4)
    for (int i = 0; i < N; i++) {
        #pragma omp critical
        {
            counter++;
        }
    }

    double elapsed = omp_get_wtime() - t0;
    printf("critical: counter = %d, time = %.4f s\n", counter, elapsed);
}

example2();

critical: counter = 1000000, time = 0.1604 s


The counter is now correct. But notice: with `critical`, threads spend most of their time **queuing at the mutex** rather than doing work. This makes parallel code *slower* than sequential for fine-grained operations.

**When to use `critical`**: When the protected block is non-trivial (multiple statements, complex logic) and cannot be expressed as a single atomic operation.

---
## 3. Fix #2 - `#pragma omp atomic`

`atomic` maps directly to a hardware atomic instruction (e.g., `LOCK XADD` on x86). It is **much faster** than `critical` for simple read-modify-write operations.

In [3]:
void example3() {
    int counter = 0;
    double t0 = omp_get_wtime();

    #pragma omp parallel for num_threads(4)
    for (int i = 0; i < N; i++) {
        #pragma omp atomic
        counter++;   // maps to a single atomic hardware instruction
    }

    double elapsed = omp_get_wtime() - t0;
    printf("atomic:   counter = %d, time = %.4f s\n", counter, elapsed);
}

example3();

atomic:   counter = 1000000, time = 0.0168 s


### `atomic` clauses (OpenMP 3.1+)

| Clause | Effect |
|--------|--------|
| `atomic update` (default) | `x++`, `x--`, `x += expr`, etc. |
| `atomic read` | `v = x` atomically |
| `atomic write` | `x = expr` atomically |
| `atomic capture` | `v = x; x op= expr` - read + update together |

**Limitation**: `atomic` only works for a *single* variable with a *single* supported operator. Use `critical` for anything more complex.

---
## 4. Fix #3 - `reduction` clause (the fastest approach)

The `reduction` clause is the idiomatic OpenMP solution for accumulation. Each thread gets a **private copy** of the variable, works independently (no synchronisation!), and OpenMP merges the results at the end of the parallel region.

This is a **divide-and-conquer** strategy: eliminate the shared state entirely during the parallel phase.

In [4]:
void example4() {
    int counter = 0;
    double t0 = omp_get_wtime();

    #pragma omp parallel for num_threads(4) reduction(+:counter)
    for (int i = 0; i < N; i++) {
        counter++;   // each thread increments its own private copy
    }

    double elapsed = omp_get_wtime() - t0;
    printf("reduction: counter = %d, time = %.4f s\n", counter, elapsed);
}

example4();

reduction: counter = 1000000, time = 0.0005 s


### Supported reduction operators

| Operator | Initial value | Use case |
|----------|--------------|----------|
| `+` | 0 | Sum |
| `*` | 1 | Product |
| `-` | 0 | Subtraction accumulation |
| `min` | type max | Minimum value |
| `max` | type min | Maximum value |
| `&` | ~0 | Bitwise AND |
| `\|` | 0 | Bitwise OR |
| `^` | 0 | Bitwise XOR |
| `&&` | 1 | Logical AND |
| `\|\|` | 0 | Logical OR |

---
## 5. Benchmark - Compare All Approaches

In [5]:
#define THREADS 4
#define RUNS    5

double bench_sequential() {
    long long c = 0;
    double t0 = omp_get_wtime();
    for (int i = 0; i < N; i++) c++;
    return omp_get_wtime() - t0;
}

double bench_critical() {
    long long c = 0;
    double t0 = omp_get_wtime();
    #pragma omp parallel for num_threads(THREADS)
    for (int i = 0; i < N; i++) {
        #pragma omp critical
        c++;
    }
    return omp_get_wtime() - t0;
}

double bench_atomic() {
    long long c = 0;
    double t0 = omp_get_wtime();
    #pragma omp parallel for num_threads(THREADS)
    for (int i = 0; i < N; i++) {
        #pragma omp atomic
        c++;
    }
    return omp_get_wtime() - t0;
}

double bench_reduction() {
    long long c = 0;
    double t0 = omp_get_wtime();
    #pragma omp parallel for num_threads(THREADS) reduction(+:c)
    for (int i = 0; i < N; i++) c++;
    return omp_get_wtime() - t0;
}

double avg(double (*fn)()) {
    double s = 0;
    for (int i = 0; i < RUNS; i++) s += fn();
    return s / RUNS;
}

printf("sequential: %.6f s\n", avg(bench_sequential));
printf("critical:   %.6f s\n", avg(bench_critical));
printf("atomic:     %.6f s\n", avg(bench_atomic));
printf("reduction:  %.6f s\n", avg(bench_reduction));

sequential: 0.001751 s
critical:   0.165203 s
atomic:     0.019443 s
reduction:  0.000474 s


### Reading the results

| Approach | Why it's slow/fast |
|----------|--------------------|
| **sequential** | Baseline: single-threaded, no sync overhead |
| **critical** | Threads serialise at the mutex - effectively sequential *plus* lock overhead |
| **atomic** | Hardware atomic instruction: much cheaper than a mutex, but still causes cache-line bouncing between cores |
| **reduction** | No synchronisation during the loop; one merge at the end - fastest of the parallel options |

---
## 6. A More Realistic Example - Histogram

Counters are a toy problem. A **histogram** is a real-world accumulation where multiple bins are updated, making `reduction` over a single variable insufficient. This shows `critical` vs `atomic` in a meaningful context.

In [6]:
#include <stdlib.h>
#include <string.h>

#define N_SAMPLES  10000000
#define N_BINS     16

// Approach A: atomic update per bin
double histogram_atomic(int *data, long long *hist) {
    memset(hist, 0, N_BINS * sizeof(long long));
    double t0 = omp_get_wtime();
    #pragma omp parallel for num_threads(THREADS)
    for (int i = 0; i < N_SAMPLES; i++) {
        #pragma omp atomic
        hist[data[i]]++;
    }
    return omp_get_wtime() - t0;
}

// Approach B: private histogram per thread, merge at end
double histogram_private(int *data, long long *hist) {
    memset(hist, 0, N_BINS * sizeof(long long));
    double t0 = omp_get_wtime();
    #pragma omp parallel num_threads(THREADS)
    {
        long long local[N_BINS] = {0};   // thread-private copy
        #pragma omp for
        for (int i = 0; i < N_SAMPLES; i++)
            local[data[i]]++;
        // Merge: each bin is independent, so atomic is fine here
        for (int b = 0; b < N_BINS; b++) {
            #pragma omp atomic
            hist[b] += local[b];
        }
    }
    return omp_get_wtime() - t0;
}

// Generate random data
int *data = (int*)malloc(N_SAMPLES * sizeof(int));
srand(42);
for (int i = 0; i < N_SAMPLES; i++) data[i] = rand() % N_BINS;

long long hist[N_BINS];

double t_atomic   = histogram_atomic(data, hist);
double t_private  = histogram_private(data, hist);

printf("atomic (per element): %.4f s\n", t_atomic);
printf("private + merge:      %.4f s  (speedup vs atomic: %.1fx)\n",
       t_private, t_atomic / t_private);

// Print histogram to verify correctness
long long total = 0;
printf("\nHistogram (should sum to %d):\n", N_SAMPLES);
for (int b = 0; b < N_BINS; b++) { printf("  bin[%2d] = %lld\n", b, hist[b]); total += hist[b]; }
printf("Total: %lld\n", total);

free(data);

atomic (per element): 0.1241 s
private + merge:      0.0066 s  (speedup vs atomic: 18.9x)

Histogram (should sum to 10000000):
  bin[ 0] = 623773
  bin[ 1] = 624866
  bin[ 2] = 624957
  bin[ 3] = 624934
  bin[ 4] = 625238
  bin[ 5] = 626471
  bin[ 6] = 625747
  bin[ 7] = 624270
  bin[ 8] = 624621
  bin[ 9] = 624898
  bin[10] = 624977
  bin[11] = 625217
  bin[12] = 624620
  bin[13] = 625424
  bin[14] = 625240
  bin[15] = 624747
Total: 10000000


The **private histogram** pattern is the general-purpose technique when you can't express the accumulation as a single built-in `reduction`. It trades memory (one copy of the histogram per thread) for speed (zero disagreement during the loop).

> **Rule of thumb**: If threads all write to the *same* bin at the same time, cache-line contention kills performance even with `atomic`. Private copies eliminate this entirely.

---
## 7. Summary - When to Use What

| Mechanism | Overhead | Flexibility | Best for |
|-----------|----------|-------------|----------|
| `reduction` | **Lowest** | Only built-in ops on loop variable | Sums, products, min/max in loops |
| `atomic` | Low | Single variable, supported operators | Simple shared counters/flags |
| `critical` | **Highest** | Arbitrary code | Complex shared-state updates |
| Private + merge | Low (memory) | Anything | Histograms, complex accumulators |